# Fase 3.4 — VS Code + Colab
Execute 1–5 em cada sessão. A célula 6 prepara dados se necessário e a 7 faz smoke test. A célula 8 inicia ou retoma o treino somente quando executada explicitamente.

In [ ]:
# 1. GPU e dependências
import torch
assert torch.cuda.is_available(), 'Conecte uma GPU do Colab'
print('GPU:', torch.cuda.get_device_name(0))
print('Verificando/instalando dependências...')
%pip install datasets scikit-learn cryptography

GPU: Tesla T4
96850719

In [ ]:
# 2. Drive e pastas persistentes; nenhuma pasta de execução anterior é alterada.
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/TCC_Final')
for folder in ('credenciais', 'modelos', 'dados_fase3_4', 'experimentos'):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print('Drive:', ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Drive: /content/drive/MyDrive/TCC_Final


In [ ]:

# 3. Desbloqueio da credencial GitHub
# Compatível com os formatos antigo e novo de github.enc

import base64
import getpass
import json

from cryptography.fernet import Fernet, InvalidToken
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.kdf.pbkdf2 import PBKDF2HMAC

credential_path = ROOT / "credenciais" / "github.enc"

if not credential_path.is_file():
    raise FileNotFoundError(
        f"Credencial não encontrada: {credential_path}"
    )

credential = json.loads(
    credential_path.read_text(encoding="utf-8")
)

# Identifique pelo nome do campo cifrado, sem exigir metadados no formato antigo.
if not isinstance(credential, dict) or "salt" not in credential:
    raise ValueError("Formato de github.enc não reconhecido.")
if "ciphertext" in credential and "token" in credential:
    raise ValueError("Credencial ambígua: dois campos cifrados.")
if "ciphertext" in credential:
    if credential.get("version") != 1:
        raise ValueError("Versão de credencial não suportada.")
    iterations = int(credential["iterations"])
    encrypted_token = credential["ciphertext"]
elif "token" in credential:
    iterations = 600000
    encrypted_token = credential["token"]

else:
    raise ValueError(
        "Formato de github.enc não reconhecido."
    )
if iterations < 600000 or not isinstance(encrypted_token, str):
    raise ValueError("Parâmetros da credencial inválidos.")

salt = base64.b64decode(credential["salt"])

password = getpass.getpass(
    "Senha de desbloqueio do GitHub: "
)

kdf = PBKDF2HMAC(
    algorithm=hashes.SHA256(),
    length=32,
    salt=salt,
    iterations=iterations
)

key = base64.urlsafe_b64encode(
    kdf.derive(password.encode())
)

try:
    token = Fernet(key).decrypt(
        encrypted_token.encode()
    ).decode()

except InvalidToken:
    raise ValueError(
        "Senha incorreta ou credencial corrompida."
    ) from None

finally:
    del password, kdf, key

print("Credencial desbloqueada nesta sessão.")


Credencial desbloqueada nesta sessão.


In [ ]:
# 4. Snapshot imutável da branch privada no runtime; não faz git clone nem toca no Drive.
import io, os, re, shutil, tempfile, time, urllib.request, urllib.error, zipfile
OWNER, REPO, BRANCH = 'vicemnovo', 'TCC_Final', 'fase3-ajuste-fino'
API = f'https://api.github.com/repos/vicemnovo/TCC_Final'
def github_read(url):
    request = urllib.request.Request(url, headers={'Authorization': f'Bearer {token}', 'Accept': 'application/vnd.github+json', 'User-Agent': 'TCC-Fase3-Colab'})
    try:
        with urllib.request.urlopen(request, timeout=120) as response:
            chunks, size, last_notice = [], 0, time.monotonic()
            while chunk := response.read(1024 * 1024):
                chunks.append(chunk)
                size += len(chunk)
                if time.monotonic() - last_notice >= 30:
                    print(f'Download GitHub em andamento: {size / 1048576:.1f} MiB', flush=True)
                    last_notice = time.monotonic()
            return b''.join(chunks)
    except urllib.error.HTTPError as error:
        raise RuntimeError(f'GitHub HTTP {error.code} ao consultar {url.split(API)[-1] or "/"}') from None
try:
    repo_info = json.loads(github_read(API))
except RuntimeError as error:
    raise RuntimeError('Repositório privado inacessível: verifique owner, seleção do repositório no token e permissão Metadata (read).') from error
try:
    branch_info = json.loads(github_read(f'{API}/branches/{BRANCH}'))
except RuntimeError as error:
    if 'HTTP 404' not in str(error):
        raise
    raise RuntimeError(f'Repositório acessível, mas a branch {BRANCH!r} não foi encontrada. Publique essa branch; a branch padrão é {repo_info["default_branch"]!r}.') from error
commit = branch_info['commit']['sha']
assert re.fullmatch(r'[0-9a-f]{40}', commit)
CODE_ROOT = Path('/content/tcc_codigo') / commit
required = ('experimentos/fase3_4_finetune.py', 'Notebooks/fase3 - openfake/fase3.3_compressao.ipynb')
if not CODE_ROOT.exists():
    print('Baixando snapshot do código...', flush=True)
    archive = github_read(f'{API}/zipball/{commit}')
    CODE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.TemporaryDirectory(prefix='tcc_sync_', dir=CODE_ROOT.parent) as temporary:
        staged = Path(temporary)
        print('Extraindo snapshot no runtime...', flush=True)
        with zipfile.ZipFile(io.BytesIO(archive)) as zf:
            last_notice = time.monotonic()
            extracted = 0
            for item in zf.infolist():
                parts = Path(item.filename).parts
                if item.is_dir() or len(parts) < 2:
                    continue
                relative = Path(*parts[1:])
                if relative.is_absolute() or '..' in relative.parts:
                    raise ValueError('Caminho inseguro no snapshot')
                destination = staged / relative
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(item) as source, destination.open('wb') as output:
                    shutil.copyfileobj(source, output)
                extracted += 1
                if time.monotonic() - last_notice >= 30:
                    print(f'Extração em andamento: {extracted} arquivos', flush=True)
                    last_notice = time.monotonic()
        if not all((staged / name).is_file() for name in required):
            raise FileNotFoundError('Publique a branch com o script e o notebook original antes de sincronizar')
        staged.rename(CODE_ROOT)
assert all((CODE_ROOT / name).is_file() for name in required)
del token
print('Código sincronizado:', commit)
print('Pasta local do runtime:', CODE_ROOT)

Código sincronizado: a54385bc74f69eb5e4bc632852cf9895474d4206
Pasta local do runtime: /content/tcc_codigo/a54385bc74f69eb5e4bc632852cf9895474d4206


In [ ]:
# 5. Versão, dependências e checkpoint inicial, sem treino.
import importlib, subprocess, sys
for module in ('torch', 'torchvision', 'datasets', 'sklearn', 'PIL', 'numpy', 'cryptography'):
    importlib.import_module(module)
SCRIPT = CODE_ROOT / 'experimentos/fase3_4_finetune.py'
SOURCE = CODE_ROOT / 'Notebooks/fase3 - openfake/fase3.3_compressao.ipynb'
assert SCRIPT.is_file() and SOURCE.is_file()
for relative in ('modelos/fase2_gate_vit_fft_cnn/phase2_final.pt', 'modelos/fase3_3_compressao_v1/phase3_final.pt'):
    assert (ROOT / relative).is_file(), f'Checkpoint ausente: {ROOT / relative}'
subprocess.run([sys.executable, '-m', 'py_compile', str(SCRIPT)], check=True)
ck = torch.load(ROOT / 'modelos/fase3_3_compressao_v1/phase3_final.pt', map_location='cpu', weights_only=False)
assert 'model_state_dict' in ck
del ck
ENV = os.environ.copy()
ENV['TCC_CODE_ROOT'] = str(CODE_ROOT)
print('Checkpoint e script verificados; commit:', commit)

# Executa o script com logs em tempo real e aviso periódico de atividade.
import queue, threading, time
def executar_com_progresso(modo, intervalo=30):
    ambiente = ENV.copy()
    ambiente['PYTHONUNBUFFERED'] = '1'
    processo = subprocess.Popen([sys.executable, '-u', str(SCRIPT), modo], env=ambiente,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
    mensagens = queue.Queue()
    def ler_saida():
        for linha in processo.stdout:
            mensagens.put(linha)
        mensagens.put(None)
    threading.Thread(target=ler_saida, daemon=True).start()
    inicio, terminou = time.monotonic(), False
    print(f'{modo}: processo iniciado (PID {processo.pid})', flush=True)
    while not terminou:
        try:
            linha = mensagens.get(timeout=intervalo)
            if linha is None:
                terminou = True
            else:
                print(linha, end='', flush=True)
        except queue.Empty:
            print(f'{modo}: em execução há {(time.monotonic() - inicio) / 60:.1f} min; aguardando próxima saída...', flush=True)
    codigo = processo.wait()
    if codigo:
        raise RuntimeError(f'{modo} falhou (código {codigo}); veja o traceback acima')
    print(f'{modo}: concluído em {(time.monotonic() - inicio) / 60:.1f} min', flush=True)

Checkpoint e script verificados; commit: a54385bc74f69eb5e4bc632852cf9895474d4206


In [ ]:
# 6. Preparação persistente; necessária apenas na primeira vez.
manifest = ROOT / 'dados_fase3_4/manifest.json'
if not manifest.is_file():
    executar_com_progresso('prepare')
else:
    print('Dados existentes preservados:', manifest)

In [ ]:
# 7. Smoke test sem treino; não repete se já houver resultado.
smoke = ROOT / 'modelos/fase3_4_finetune_v1_smoke/smoke.json'
if not smoke.is_file():
    executar_com_progresso('smoke')
else:
    print('Smoke anterior preservado:', smoke)

In [ ]:
# 8. EXECUÇÃO MANUAL: inicia ou retoma conforme last.pt; não rode até querer treinar.
# executar_com_progresso('auto')

In [ ]:
# 9. Leitura de resultados após conclusão; não reavalia nem altera o teste.
result = ROOT / 'modelos/fase3_4_finetune_v1/final_metrics.json'
print(json.dumps(json.loads(result.read_text(encoding='utf-8')), indent=2) if result.is_file() else 'Ainda sem resultados finais.')